# 🔎 Experiment 00 — Inference & Submission

Loads the LoRA adapter saved by `01_train.ipynb` from `ARTIFACT_DIR`, applies it
to the base Gemma 3n model, then:
1. (optional) evaluates WER/CER per language,
2. generates `submission.csv` for the Zindi leaderboard.

**No training happens here.** The only link to the training notebook is
`ARTIFACT_DIR` — it must point at the same folder training wrote to.

## 1. Install dependencies

In [ ]:
%%capture
!pip install -q \
    transformers>=4.52.0 \
    datasets \
    peft \
    trl \
    jiwer \
    timm \
    librosa \
    soundfile \
    accelerate \
    bitsandbytes \
    huggingface_hub \
    torchao \
    pandas

print("✅ Dependencies installed.")

## 2. Configuration  
_Must match the training notebook (same `MODEL_ID`, prompts, `SAMPLE_RATE`, `ARTIFACT_DIR`)._

In [ ]:
import dataclasses
from typing import Final

# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------
#   "google/gemma-3n-E2B-it"  — ~2B effective params (recommended for T4)
#   "google/gemma-4n-E4B-it"  — ~4B effective params (recommended for A100)
MODEL_ID: Final[str] = "google/gemma-3n-E2B-it"

# ---------------------------------------------------------------------------
# Dataset
# ---------------------------------------------------------------------------
DATASET_ID: Final[str] = "google/WaxalNLP"

# This challenge scores Lingala, Shona and Luganda. We train ONE multilingual
# model on all three, because Phase 2 provides no language metadata — the model
# must transcribe from audio alone.
LANGUAGES: Final[list[str]] = ["lin", "sna", "lug"]

# ---------------------------------------------------------------------------
# Prompts (MUST be identical in train + inference)
# ---------------------------------------------------------------------------
SYSTEM_MESSAGE: Final[str] = (
    "You are an assistant that transcribes speech accurately."
)
USER_MESSAGE: Final[str] = "Please transcribe this audio."

# ---------------------------------------------------------------------------
# Audio
# ---------------------------------------------------------------------------
SAMPLE_RATE: Final[int] = 16_000

# ---------------------------------------------------------------------------
# Artifacts — THE coupling point between the two notebooks.
# Training writes the LoRA adapter + processor here; inference reads from here.
# Point this at persistent storage (Drive / a bucket) so it survives across
# separate Colab sessions.
# ---------------------------------------------------------------------------
ARTIFACT_DIR: Final[str] = "./gemma3n-asr-multilingual"

RANDOM_SEED: Final[int] = 42

print(f"Base model : {MODEL_ID}")
print(f"Languages  : {LANGUAGES}")
print(f"Loading adapter from: {ARTIFACT_DIR}")

## 3. Authenticate with Hugging Face

In [ ]:
from huggingface_hub import login

# Gemma is a gated model — accept its license on the model card, then log in
# with a read token: https://huggingface.co/settings/tokens
login()

## 4. Load base model + trained LoRA adapter

In [ ]:
import timm  # Required by Gemma3n; import before transformers loads it
import torch
import transformers
import peft

# Processor comes from the artifact dir (falls back to base model if absent).
try:
    processor = transformers.AutoProcessor.from_pretrained(ARTIFACT_DIR)
except Exception:
    processor = transformers.AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "right"

print(f"Loading base model {MODEL_ID} …")
base_model = transformers.Gemma3nForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto",
)

print(f"Applying LoRA adapter from {ARTIFACT_DIR} …")
model = peft.PeftModel.from_pretrained(base_model, ARTIFACT_DIR)
model.eval()
print(f"✅ Model ready — device: {model.device}, dtype: {model.dtype}")

## 5. Formatting + inference helpers

In [ ]:
from collections.abc import Mapping, Sequence
from typing import Any

import numpy as np
import librosa
import datasets


def audio_to_array(audio: Any) -> np.ndarray:
    """Returns a mono 1-D float32 array from a datasets Audio value.

    Works with both the classic dict form ({'array', 'sampling_rate'}) and the
    torchcodec ``AudioDecoder`` returned by datasets >= 4.
    """
    sr = SAMPLE_RATE
    if isinstance(audio, Mapping):
        arr = np.asarray(audio["array"], dtype=np.float32)
        sr = int(audio.get("sampling_rate", SAMPLE_RATE))
    elif hasattr(audio, "get_all_samples"):          # torchcodec AudioDecoder
        samples = audio.get_all_samples()
        arr = np.asarray(samples.data, dtype=np.float32)
        sr = int(getattr(samples, "sample_rate", SAMPLE_RATE))
    else:
        arr = np.asarray(audio, dtype=np.float32)
    if arr.ndim > 1:                                 # downmix channels to mono
        arr = arr.mean(axis=0)
    arr = arr.flatten()
    if sr != SAMPLE_RATE:
        arr = librosa.resample(arr, orig_sr=sr, target_sr=SAMPLE_RATE)
    return arr


def format_for_chat(
    example: Mapping[str, Any],
    system_message: str = SYSTEM_MESSAGE,
    user_message: str = USER_MESSAGE,
) -> dict[str, Any]:
    """Converts a decoded example into a chat-formatted message list."""
    audio_array = audio_to_array(example["audio"])
    return {
        **example,
        "messages": [
            {"role": "system",
             "content": [{"type": "text", "text": system_message}]},
            {"role": "user",
             "content": [
                 {"type": "audio", "audio": audio_array},
                 {"type": "text", "text": user_message},
             ]},
            {"role": "assistant",
             "content": [{"type": "text", "text": str(example["transcription"])}]},
        ],
    }


def format_batch(batch: dict[str, list[Any]]) -> dict[str, list[Any]]:
    """Formats a batch of decoded examples into chat messages."""
    num = len(batch["transcription"])
    examples = [{k: batch[k][i] for k in batch} for i in range(num)]
    formatted = [format_for_chat(ex) for ex in examples]
    return {k: [ex[k] for ex in formatted] for k in formatted[0]}


def load_split(language: str, split: str) -> datasets.IterableDataset:
    """Loads a single language/split, resampled to SAMPLE_RATE.

    NOTE: it does NOT apply the chat-formatting map. Callers map AFTER any
    ``interleave_datasets`` call, because interleave must infer Arrow features
    and cannot do so from raw/decoded audio objects (torchcodec AudioDecoder).
    """
    ds = datasets.load_dataset(
        DATASET_ID, name=f"{language}_asr", split=split, streaming=True
    )
    return ds.cast_column("audio", datasets.Audio(sampling_rate=SAMPLE_RATE))


import jiwer


def transcribe_batch(batch, model, proc, device, max_new_tokens: int = 128):
    """Runs inference on a batch and returns decoded transcriptions."""
    messages_list = [msgs[:-1] for msgs in batch["messages"]]
    audio_list = [audio_to_array(a) for a in batch["audio"]]
    text_prompts = proc.tokenizer.apply_chat_template(
        messages_list, add_generation_prompt=True, tokenize=False
    )
    inputs = proc(text=text_prompts, audio=audio_list,
                  return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            pad_token_id=proc.tokenizer.pad_token_id,
        )
    input_len = inputs.input_ids.shape[1]
    decoded = proc.tokenizer.batch_decode(
        outputs[:, input_len:], skip_special_tokens=True
    )
    return [t.strip() for t in decoded]

## 6. (Optional) Evaluate WER / CER per language

Uses the WaxalNLP `test` split (which has ground-truth transcriptions) to sanity
-check the model. This is *not* the Zindi leaderboard score, but tracks with it.
Skip if you only need a submission.

In [ ]:
def run_evaluation(eval_dataset, model, proc, num_samples=200, batch_size=4):
    """Evaluates model on eval_dataset and returns WER / CER."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    references, predictions = [], []
    for batch in eval_dataset.take(num_samples).batch(batch_size=batch_size):
        preds = transcribe_batch(batch, model, proc, device)
        references.extend(batch["transcription"])
        predictions.extend(preds)
    refs = [r.lower() for r in references]
    preds = [p.lower() for p in predictions]
    return {"wer": jiwer.wer(refs, preds), "cer": jiwer.cer(refs, preds)}


all_metrics = {}
for lang in LANGUAGES:
    test_ds = load_split(lang, "test").map(format_batch, batched=True, batch_size=32)
    print(f"Evaluating {lang} …")
    m = run_evaluation(test_ds, model, processor, num_samples=200)
    all_metrics[lang] = m
    print(f"   {lang}: WER {m['wer']:.2%} | CER {m['cer']:.2%}")

mean_wer = sum(m["wer"] for m in all_metrics.values()) / len(all_metrics)
mean_cer = sum(m["cer"] for m in all_metrics.values()) / len(all_metrics)
print(f"\n📊 Mean: WER {mean_wer:.2%} | CER {mean_cer:.2%}")
print(f"   Leaderboard-style (0.5·WER + 0.5·CER): "
      f"{0.5 * mean_wer + 0.5 * mean_cer:.4f}")

## 7. Generate the Zindi submission

Writes `submission.csv` with columns `ID,Target`, one row per clip in
`SampleSubmission.csv`. The clip id field is auto-detected from the dataset.

> **Every row must be filled** — the `missing` counter must be 0 before you
> submit. We match by id, not by the language prefix (no metadata in Phase 2).

In [ ]:
import pandas as pd

# Point this at the competition's SampleSubmission.csv.
SAMPLE_SUBMISSION_PATH = (
    "../../data/google-waxal-asr-challenge20260630-10570-elxebu/SampleSubmission.csv"
)
SUBMISSION_OUTPUT_PATH = "submission.csv"


def _detect_id_key(example, wanted_ids):
    """Finds which dataset field holds the Zindi clip ID (e.g. 'lug_96114')."""
    for key in ("id", "ID", "utt_id", "utterance_id", "file", "path", "name"):
        if key in example and str(example[key]) in wanted_ids:
            return key
    for key, value in example.items():
        if isinstance(value, str) and value in wanted_ids:
            return key
    raise KeyError(
        "No ID field matched SampleSubmission IDs. "
        f"Example keys: {list(example.keys())}"
    )


sample_sub = pd.read_csv(SAMPLE_SUBMISSION_PATH)
wanted_ids = set(sample_sub["ID"].astype(str))
print(f"Sample submission requires {len(wanted_ids)} predictions.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
predictions, id_key = {}, None

for lang in LANGUAGES:
    test_ds = load_split(lang, "test").map(format_batch, batched=True, batch_size=32)
    if id_key is None:
        id_key = _detect_id_key(next(iter(test_ds)), wanted_ids)
        print(f"Using dataset id field: '{id_key}'")

    print(f"Transcribing {lang} …")
    for batch in test_ds.batch(batch_size=4):
        ids = [str(x) for x in batch[id_key]]
        if not any(i in wanted_ids and i not in predictions for i in ids):
            continue
        preds = transcribe_batch(batch, model, processor, device)
        for _id, pred in zip(ids, preds):
            if _id in wanted_ids:
                predictions[_id] = pred

sample_sub["Target"] = sample_sub["ID"].astype(str).map(predictions).fillna("")
missing = int((sample_sub["Target"] == "").sum())
print(f"Filled {len(sample_sub) - missing}/{len(sample_sub)} rows; {missing} missing.")
if missing:
    print("⚠️ Empty rows remain — check the id field / test coverage before submitting.")

sample_sub.to_csv(SUBMISSION_OUTPUT_PATH, index=False)
print(f"✅ Wrote {SUBMISSION_OUTPUT_PATH}")
sample_sub.head()